# Capstone Topic 5: Final Interview Preparation

> **Phase 7 → Week 16 (Capstone) → Topic 5**

This notebook is your interview playbook — complete reference answers for every major topic across all 16 weeks, organized by interview category. No code to run; read, internalize, practice out loud.

---

## How to Use This Notebook

1. Read each question. Cover the answer. Say your answer out loud.
2. Uncover and compare. Note gaps.
3. For system design questions: draw the diagram on paper before reading the answer.
4. For coding questions: write the code in a new cell before reading.

Target: be able to answer any question cold, in under 2 minutes, without notes.

---
## Category 1: Spark Internals

In [ ]:
print("""
Q: Explain the Spark execution model: jobs, stages, tasks.
A: An action (count, write) triggers a Job. Spark's DAG scheduler splits the job into Stages at
   shuffle boundaries (groupBy, join, repartition). Each stage consists of Tasks — one task per
   input partition. Tasks are the unit of parallelism; each task runs on one executor core.
   Wide transformations (shuffle) create new stages because data must cross the network.
   Narrow transformations (map, filter) stay in the same stage — no shuffle.

Q: What is the difference between a transformation and an action?
A: Transformations (select, filter, join, groupBy) are lazy — they build a query plan but execute
   nothing. Actions (count, collect, write, show) trigger execution. This laziness lets Spark
   optimize the full plan (predicate pushdown, filter reordering) before running anything.

Q: What causes a shuffle and why is it expensive?
A: Wide transformations that require data from multiple partitions: groupBy, join, repartition,
   distinct, orderBy. Expensive because: (1) data is serialized and written to disk (shuffle write),
   (2) transferred over the network between executors, (3) deserialized on the receiving side
   (shuffle read). GC pressure and sort cost add up. For 1 TB data, a bad shuffle can be 10×
   slower than a local computation.

Q: What is data skew and how do you fix it?
A: Skew: one partition has far more data than others — one task takes 10× longer (straggler).
   Cause: groupBy/join on a column with highly skewed values (e.g., one user_id has 50% of orders).
   Fixes:
   1. Salting: add a random suffix to the key (key + "_" + rand(N)) → spread across N partitions
      → aggregate by real key in a second pass.
   2. Broadcast join: if one side is small (<= spark.sql.autoBroadcastJoinThreshold, default 10MB),
      Spark broadcasts it to all executors — no shuffle at all.
   3. AQE skew join: Spark 3.x auto-detects and splits skewed partitions. Enable with
      spark.sql.adaptive.skewJoin.enabled=true.

Q: What is the difference between cache() and persist()?
A: cache() = persist(StorageLevel.MEMORY_AND_DISK) — first tries memory, spills to disk.
   persist() lets you specify: MEMORY_ONLY (fastest, OOM risk), MEMORY_AND_DISK (safe default),
   DISK_ONLY (slowest, saves memory), OFF_HEAP (for large datasets with Tungsten).
   Use cache/persist when a DataFrame is used multiple times in the same job. Never cache a
   streaming DataFrame or one that's only used once.

Q: What is AQE (Adaptive Query Execution)?
A: Spark 3.0+ feature that re-optimizes the plan at runtime using statistics collected during
   execution. Three key optimizations:
   1. Coalesce shuffle partitions: 200 partitions → actual data size → reduce to N (avoids small files)
   2. Switch join strategies: if one side turns out small at runtime → convert to broadcast join
   3. Skew join optimization: detect and split skewed partitions automatically
   Enable: spark.sql.adaptive.enabled=true (default in Spark 3.2+).
""")

---
## Category 2: Delta Lake

In [ ]:
print("""
Q: How does Delta Lake guarantee ACID transactions on a distributed file system like S3?
A: Via the transaction log (_delta_log/). Every write creates a new JSON commit file atomically
   (S3 PUT is atomic). Readers read the latest committed log entry — they never see partial writes.
   Concurrent writes use optimistic concurrency: each writer reads the current version, writes its
   data files, then attempts to commit. If another writer committed first, Delta detects the conflict
   and retries. The log is the source of truth; Parquet files are immutable data.

Q: What is the difference between MERGE INTO and overwrite partitions?
A: MERGE INTO: row-level upsert (UPDATE + INSERT, optionally DELETE) based on a match condition.
   Best for CDC (update existing records, insert new ones). Rewrites only affected files.
   Partition overwrite (mode=overwrite + dynamicPartitionOverwrite): replaces entire partitions
   atomically. Best for idempotent daily batch loads where you recompute the full partition.
   MERGE is more precise but slower (must scan for matches); overwrite is coarser but faster.

Q: How does time travel work in Delta Lake? When would you use it?
A: The transaction log tracks every file added/removed per version. Querying version N → Delta
   reads the log up to version N and only opens files that existed at that version.
   Syntax: .option("versionAsOf", N) or .option("timestampAsOf", "2024-01-15")
   Uses: (1) audit/debug: compare today's data to yesterday's, (2) recovery: restore after bad
   write (RESTORE TO VERSION N), (3) regulatory: snapshot for compliance reporting.
   Limit: VACUUM removes old files — time travel only works back to the retention window (default 7d).

Q: What is Change Data Feed (CDF) and when would you use it?
A: CDF tracks row-level changes (insert, update_preimage, update_postimage, delete) as a stream
   queryable via readChangeFeed=true. Use it when downstream systems need to know WHAT changed,
   not just the current state. Use cases: (1) incremental Silver → Gold without re-reading all
   Silver, (2) propagate GDPR deletes downstream without full reprocess, (3) CDC to Kafka for
   event-driven consumers. Enable per table: ALTER TABLE SET TBLPROPERTIES (delta.enableChangeDataFeed=true).

Q: What is OPTIMIZE and ZORDER? How do they improve query performance?
A: OPTIMIZE: compacts small Parquet files into larger ones (target ~1 GB). Solves the small file
   problem from streaming or frequent small batch writes. Reduces the number of S3 LIST/GET calls.
   ZORDER BY col1, col2: co-locates related data within files. If you always filter by region,
   ZORDER BY region ensures each file spans a narrow range of regions → Parquet stats skip
   irrelevant files entirely (data skipping). Best column to ZORDER: highest cardinality column
   used most in WHERE clauses. Max 2-3 columns (diminishing returns beyond that).
""")

---
## Category 3: Streaming

In [ ]:
print("""
Q: What is exactly-once processing in Spark Structured Streaming?
A: Guarantee that each record is processed exactly once — no duplicates, no data loss.
   Spark achieves this via:
   1. Idempotent sinks: Delta Lake MERGE, Kafka transactional producers
   2. Checkpointing: offset tracking persisted to durable storage (S3/HDFS) — on restart,
      Spark reads from the last committed offset, not from the beginning
   3. WAL (Write-Ahead Log): optional for non-idempotent sinks
   Exactly-once = at-least-once delivery + idempotent sink. The source may redeliver
   (Kafka can replay offsets), but the sink handles duplicates.

Q: What is a watermark? How do you choose the watermark duration?
A: Watermark declares the max expected lateness of events. Spark will wait up to watermark
   duration before finalizing a window and dropping its state.
   Formula: watermark_threshold = max(event_time) - watermark_duration
   Events earlier than this threshold are dropped.
   Choosing duration: look at your P99 event lateness (e.g., from CloudWatch logs).
   If 99% of events arrive within 5 minutes of event time, use 10 minutes (2× P99 for safety).
   Tighter watermark = less memory, more dropped events. Looser = more memory, higher accuracy.

Q: What is the difference between outputMode append, update, and complete?
A: append: only new rows since last trigger. For windowed aggregations: only finalized windows
   (past watermark). Cannot write non-windowed aggregations in append mode.
   update: rows that changed since last trigger (updated aggregates). Like a database CDC stream.
   complete: the full result set every trigger. Only practical for small aggregates
   (e.g., global count) — rewrites everything each time.
   Most common production choice: append with watermark for windowed aggregates → Delta table.

Q: What is foreachBatch? Give a production use case.
A: foreachBatch(func) calls func(batch_df, batch_id) for each micro-batch, giving you a regular
   DataFrame to work with (no streaming API restrictions). Use cases:
   1. Delta MERGE INTO (not natively supported in streaming sink)
   2. Write to multiple sinks atomically (e.g., Delta + send Kafka alert)
   3. Run DQ checks per batch and fail the stream if DQ fails
   4. Custom deduplication logic that requires looking up existing records
   Idempotency: batch_id is stable on retry — use it to make the function idempotent.

Q: Kafka source: what is the difference between startingOffsets=earliest vs latest?
A: earliest: read all historical messages from the beginning of the topic.
   Use when: first time running, need to backfill all historical data.
   Risk: can replay millions of old events if topic has long retention.
   latest: start from the current end of the topic — only new messages.
   Use when: restarting an existing job (checkpoint handles offset tracking anyway).
   In practice: set startingOffsets=latest for the first run in prod;
   after that, the checkpoint file manages offsets exactly — startingOffsets is ignored.
""")

---
## Category 4: Architecture & System Design

In [ ]:
print("""
Q: Describe the Medallion Architecture. What goes in each layer?
A: Three-layer pattern for data lake organization:
   Bronze (raw): exact copy of source data, as-is, with ingestion metadata (_ingested_at,
   _source, _run_id). Never modified. Partitioned by date. Append-only (except GDPR deletes).
   Silver (clean): validated, deduplicated, type-cast, enriched data. Dead-letter routing for
   invalid records. Partitioned by date. Business entities: orders, users, products.
   Gold (aggregated): business-specific metrics, joined and aggregated for specific use cases.
   One Gold table = one business question (revenue by region, daily active users). Partitioned
   by date. What BI tools and analysts query directly.
   Key benefit: each layer is independent — you can reprocess Silver from Bronze, or Gold from
   Silver, without touching upstream data.

Q: When would you choose Delta Lake vs Apache Iceberg vs Apache Hudi?
A: Delta Lake: choose when compute is Spark on EMR or Databricks. Best Spark integration,
   MERGE is most battle-tested, OPTIMIZE + ZORDER for data skipping. CDF for CDC propagation.
   Iceberg: choose when you need engine-agnostic (Flink + Spark + Trino all reading same table),
   or for hidden partitioning (partition evolution without reader changes). Netflix/Apple scale.
   Hudi: choose when your primary use case is record-level upserts (COW vs MOR modes),
   especially for high-frequency CDC from RDS. Streaming-first design.
   For most AWS shops with Spark: Delta Lake is the safe default.

Q: How would you handle a GDPR deletion request in a data lake?
A: Three phases:
   1. Identify: query deletion_requests table for user_id. Get list of all affected tables
      (Bronze, Silver, Gold, dead-letter, streaming tables).
   2. Delete: for each table, run:
      DELETE FROM delta_table WHERE user_id = '<user_id>'
      Delta supports row-level DELETE via predicate. This marks rows as deleted in the log
      but data files still exist until VACUUM.
   3. Vacuum: set deletedFileRetentionDuration=0h (or minimum), run VACUUM delta_table.
      This physically removes Parquet files containing deleted rows.
   4. Audit: log each delete operation with timestamp, user_id, table_name.
      Verify with SELECT COUNT(*) WHERE user_id = '<user_id>' = 0.
   SLA: complete within 48 hours. Streaming: add user_id to a blocklist — filter at Silver.

Q: What is the small file problem and how do you fix it?
A: Small files: too many tiny Parquet files (< 128 MB). Causes: streaming writes many small files
   per trigger, or partitioned batch writes with high cardinality partition keys.
   Impact: Parquet file metadata overhead, S3 LIST latency, Spark task overhead (one task per file),
   slow query planning.
   Fixes:
   1. OPTIMIZE (Delta): periodic compaction job → merges small files into 1 GB target.
   2. coalesce() / repartition() before write: control output file count explicitly.
   3. Increase trigger interval: streaming batches write fewer, larger files.
   4. AQE coalesceShuffle: automatically reduces shuffle partitions → fewer output files.

Q: How do you design for idempotency in a batch pipeline?
A: Idempotency = running the pipeline twice for the same input produces the same output.
   Techniques:
   1. Dynamic partition overwrite: spark.sql.sources.partitionOverwriteMode=dynamic
      → only the date partition being processed is replaced.
   2. Delta MERGE INTO: upserts are naturally idempotent — second run matches all existing rows
      and updates them to the same values.
   3. Fresh cluster per run: no shared in-memory state between runs.
   4. Metrics overwrite: DQ JSON is overwritten with identical content on re-run.
   5. Dead-letter append is okay: accumulates rejections from all runs — useful for auditing.
   Test: run the same DAG execution twice. Assert row count is identical.
""")

---
## Category 5: Testing, DQ & Production

In [ ]:
print("""
Q: How do you unit test a PySpark transformation function?
A: 1. Extract the transformation as a pure function (no I/O, takes DataFrame, returns DataFrame).
   2. In conftest.py: @pytest.fixture(scope='session') def spark(): ... — one JVM for all tests.
   3. In the test: create minimal inline data (3-10 rows), call the function, assert on .collect().
   4. Test: happy path, empty DataFrame, null values, boundary values (tier thresholds), duplicates.
   5. Use chispa.assert_df_equality(actual, expected, ignore_row_order=True) for clear diff output.
   Key: test pure functions only — no writing to files, no Kafka, no S3. Those are integration tests.

Q: What are the five dimensions of data quality?
A: Completeness: no unexpected nulls (critical columns like order_id, user_id must be 100% not null).
   Validity: values within expected domain (status in {'shipped','pending','cancelled'}, amount > 0).
   Uniqueness: primary keys not duplicated (order_id is unique per batch).
   Consistency: referential integrity (every user_id in orders exists in users table).
   Timeliness: data is fresh (Gold updated within 2 hours of event time; no stale partitions).

Q: What is the dead-letter pattern? Why use it instead of filtering bad records?
A: Filtering discards invalid records permanently — you lose visibility into what failed and why.
   Dead-letter: route invalid records to a quarantine table with a _rejection_reason column.
   Benefits: (1) audit trail for compliance, (2) upstream team can fix the source and replay,
   (3) track rejection rates over time (spike = new upstream bug), (4) no data loss.
   Implementation: compute _rejection_reason as concatenation of failing checks, split into
   valid (reason == '') and dead_letter (reason != ''), write dead_letter with mode=append.

Q: What makes a Spark pipeline production-ready? (Give 6 points)
A: 1. Testable: core logic in pure functions with pytest coverage >= 80%.
   2. Idempotent: safe to re-run for the same date without duplicating data.
   3. Observable: structured JSON logging with run_id, layer, rows, duration; DQ metrics in S3;
      CloudWatch alarms on row count drops and freshness violations.
   4. Configurable: per-env config dict (dev/staging/prod) + SSM for sensitive values.
      No hardcoded paths, credentials, or cluster sizes.
   5. Exits non-zero on failure: sys.exit(1) → EMR step FAILED → Airflow detects → retries/alerts.
   6. CI/CD: lint (ruff) + test (pytest) on every PR; deploy on merge with OIDC (no stored keys).

Q: What is GitHub OIDC and why is it better than storing AWS credentials in GitHub Secrets?
A: Stored credentials (AWS_ACCESS_KEY_ID): long-lived, leak risk, manual rotation.
   OIDC: GitHub acts as Identity Provider. On each workflow run, GitHub issues a short-lived
   signed token. Workflow presents it to AWS STS → STS verifies GitHub's signature → returns
   temporary credentials (expire in 1 hour). No long-lived secrets stored anywhere.
   If credentials leak: they expire automatically. No emergency rotation needed.
   Setup: create OIDC provider in IAM, create role with trust policy restricting to specific
   repo/branch, add permissions: id-token: write to workflow.
""")

---
## Category 6: Performance Optimization

In [ ]:
print("""
Q: A Spark job is running slowly. Walk me through your debugging process.
A: Structured approach:
   1. Spark UI → Jobs tab: find the slowest stage. Note input bytes, shuffle bytes, task count.
   2. Stage detail → Task tab: look for stragglers (one task much slower than others = skew).
      Check: GC time > 10% of task time = OOM pressure. Shuffle spill = memory too small.
   3. Check data skew: spark.sql.adaptive.skewJoin.enabled=true may fix automatically.
      Or inspect the groupBy/join key distribution manually.
   4. Check shuffle partitions: spark.sql.shuffle.partitions=200 (default). If data is small,
      reduce to 10-20. If data is large and tasks are slow, increase to 500+.
   5. Check broadcast joins: if joining a small table (< 100 MB), add broadcast hint:
      df.join(broadcast(small_df), 'key').
   6. Check I/O: filter early (predicate pushdown), select only needed columns (projection
      pushdown), partition-prune (filter on partition column = skip reading entire partitions).

Q: When should you use repartition() vs coalesce()?
A: repartition(N): full shuffle — distributes data evenly across N partitions. Use when:
   you need to increase partition count, or fix severe skew, or repartition by a specific column
   (repartition(N, 'key')) for better downstream join performance.
   coalesce(N): no shuffle — merges partitions locally. Use only to DECREASE partition count,
   typically before a write to avoid small files. Never use coalesce() to increase partitions.
   Rule: coalesce() is always cheaper than repartition() — only use repartition() when you
   actually need a full redistribution.

Q: What is predicate pushdown and how does Spark apply it to Delta tables?
A: Predicate pushdown: push filter conditions as close to the data source as possible, so
   less data is read into memory.
   Parquet level: Parquet stores min/max stats per column per row group. Spark compares the
   filter predicate against these stats — if the predicate can't match any value in the row
   group, the entire row group is skipped (no disk read).
   Delta level: partition pruning — if you filter on the partition column (date='2024-01-15'),
   Delta only lists/reads files in that partition directory. Eliminates S3 LIST calls for
   irrelevant partitions.
   ZORDER level: co-locates related data → more files skipped per predicate.
   Combined: Delta + ZORDER + Parquet stats = data skipping at partition, file, and row group level.

Q: What is broadcast join and when is it safe to use?
A: A small table is broadcast (sent in full) to every executor, so the join is done locally
   without shuffling the large table. Zero network transfer for the large side.
   Safe when: the broadcasted table fits in executor memory. Rule of thumb: < 10% of executor
   memory, max ~2-3 GB in practice. Auto-triggered at < spark.sql.autoBroadcastJoinThreshold
   (default 10 MB). Override: join(broadcast(dim), 'key').
   Avoid when: the 'small' table is actually large and you risk OOM on executors.
   Typical use: dimension tables (products, regions, categories) joined to large fact tables.
""")

---
## Category 7: Cloud & Orchestration

In [ ]:
print("""
Q: EMR vs Glue vs Databricks — when would you choose each?
A: EMR: choose when you need cost control (Spot instances, 60-80% savings), custom Spark configs,
   Spark UI for debugging, specific library versions, or per-job cluster lifecycle.
   Best for: large-scale batch, cost-sensitive, experienced Spark teams.
   Glue: choose when you want zero infra management, pay-per-DPU, native Glue Catalog integration,
   or Lambda-like trigger from S3 events. Best for: simple ETL, small team, S3 → Redshift patterns.
   Databricks: choose when you want collaborative notebooks, Unity Catalog governance, DLT,
   MLflow, or Photon engine performance. Best for: data + ML teams, Databricks-native shops.
   Cost at ShopStream scale (500K orders/day): EMR ~$1K/month, Glue ~$3K, Databricks ~$4K+.

Q: What is the difference between Airflow sensor poke mode vs reschedule mode?
A: poke: sensor holds its worker slot while waiting. Polls every poke_interval seconds.
   Problem: if 10 sensors are all poking, they occupy 10 worker slots continuously → Celery
   workers starve; other tasks queue up.
   reschedule: sensor releases its worker slot between pokes, rescheduling itself after
   each interval. Much more efficient for long waits (hours).
   Rule: always use mode='reschedule' for sensors with timeout > 5 minutes.
   poke is only acceptable for short waits (< 1 minute) where the context switch overhead
   of reschedule exceeds the wait itself.

Q: What is catchup=False in Airflow? When would you enable catchup=True?
A: catchup=False: when a DAG is unpaused or its start_date is in the past, Airflow only
   triggers the most recent scheduled run — it does NOT backfill all missed runs.
   catchup=True (default): Airflow backfills every missed run since start_date.
   Use catchup=True when: historical data is available and you want to backfill all partitions
   (e.g., initial load of a new Gold table from 2 years of Silver data).
   Use catchup=False when: you only need daily fresh data, historical data isn't in the source
   (streaming), or a sudden unpausing shouldn't trigger 30 days of job runs.

Q: How do you handle EMR cluster failures in Airflow?
A: The critical pattern: EmrTerminateJobFlowOperator with trigger_rule=TriggerRule.ALL_DONE.
   This means 'terminate even if upstream steps failed.' Without this, a failed step leaves
   the cluster running and billing — orphaned cluster.
   Additional patterns:
   1. retries=1 on the DAG task: Airflow retries the full EMR step sequence.
   2. ActionOnFailure=CONTINUE on EMR steps: step fails, cluster continues so subsequent
      steps (freshness check) still run.
   3. Airflow SLA: if DAG doesn't complete within 2h, SLA miss callback fires PagerDuty.
   4. CloudWatch alarm: EMR Step FAILED state → SNS → Slack/PagerDuty independently of Airflow.

Q: What is S3 dynamic partition overwrite and why does it matter?
A: By default, mode=overwrite on a partitioned write replaces the ENTIRE table.
   With spark.sql.sources.partitionOverwriteMode=dynamic, only the partitions present in the
   DataFrame being written are replaced. Other partitions are untouched.
   Example: writing Silver for 2024-01-15 → only the date=2024-01-15 partition is replaced;
   date=2024-01-14 is unchanged. This enables idempotent daily batch loads.
   Without it: re-running for one day wipes all other days — catastrophic.
""")

---
## Category 8: Coding Questions

Practice these by writing the code from scratch before reading the answer.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = SparkSession.builder \
    .appName("Interview Coding") \
    .master("local[2]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

orders = spark.createDataFrame([
    ("O1", "U1", "US", 500.0, "shipped"),
    ("O2", "U1", "US", 200.0, "pending"),
    ("O3", "U2", "EU", 1500.0, "shipped"),
    ("O4", "U3", "US", 100.0, "cancelled"),
    ("O5", "U2", "EU", 300.0, "shipped"),
    ("O6", "U4", "APAC", 800.0, "shipped"),
], ["order_id", "user_id", "region", "amount", "status"])

# ── Q1: Top 2 orders by amount per region (window function) ────
print("Q1: Top 2 orders by amount per region")
w = Window.partitionBy("region").orderBy(F.col("amount").desc())
top2 = orders.withColumn("rank", F.rank().over(w)) \
             .filter(F.col("rank") <= 2) \
             .drop("rank")
top2.orderBy("region", F.col("amount").desc()).show()

# ── Q2: Revenue by region, only shipped orders ─────────────────
print("Q2: Revenue by region (shipped only)")
orders.filter(F.col("status") == "shipped") \
      .groupBy("region") \
      .agg(F.sum("amount").alias("revenue"),
           F.count("*").alias("order_count")) \
      .orderBy(F.col("revenue").desc()) \
      .show()

# ── Q3: Running total per user ordered by amount ───────────────
print("Q3: Running total per user")
w2 = Window.partitionBy("user_id").orderBy("amount").rowsBetween(Window.unboundedPreceding, 0)
orders.withColumn("running_total", F.sum("amount").over(w2)) \
      .select("order_id", "user_id", "amount", "running_total") \
      .orderBy("user_id", "amount") \
      .show()

# ── Q4: Detect duplicate order_ids ────────────────────────────
print("Q4: Duplicate order detection")
duped = orders.union(
    spark.createDataFrame([("O1", "U1", "US", 500.0, "shipped")],
                          ["order_id","user_id","region","amount","status"])
)
dupes = duped.groupBy("order_id").count().filter(F.col("count") > 1)
print(f"Duplicate order_ids found: {dupes.count()}")
dupes.show()

---
## Category 9: Behavioral / Situational

In [ ]:
print("""
Q: A data engineer on your team pushed a change that overwrote 3 days of Gold data with
   incorrect aggregations. How do you recover?
A: Delta Lake time travel:
   1. Identify the last known good version:
      spark.sql("DESCRIBE HISTORY delta.`s3://shopstream-data/gold/revenue_by_region`").show()
   2. Verify the good version:
      spark.read.format('delta').option('versionAsOf', N).load(path).show()
   3. Restore:
      spark.sql(f"RESTORE TABLE delta.`{path}` TO VERSION AS OF {N}")
   4. Communicate: notify downstream consumers (QuickSight, analysts) of the outage and recovery.
   5. Post-mortem: add CI test that verifies Gold row count != 0 after write; add PR review
      requirement for Gold-layer changes; consider making Gold tables read-only by default.
   Prevention: RESTORE is fast (metadata only), but add a data contract: Gold row count must
   be within ±30% of previous day, else fail the task.

Q: The Silver job is taking 4 hours instead of the usual 45 minutes. What do you do?
A: Structured diagnosis:
   1. Check Spark UI: which stage is slow? What's the shuffle data size?
   2. Check task metrics: are there straggler tasks (one core task running 3h, others done)?
      → Data skew. Fix: AQE skew join, or salting.
   3. Check executor logs: OOM errors? → Increase executor memory or reduce broadcast join
      threshold.
   4. Check input data: is today's input 10× larger than usual? → Upstream data volume spike.
      Alert upstream team. Scale up task nodes (add more SPOT nodes via EMR Managed Scaling).
   5. Check shuffle partitions: if data grew 10×, spark.sql.shuffle.partitions=200 creates
      tasks with 5× more data each → increase to 1000.
   Immediate action: if SLA is breached, notify stakeholders. Run with larger SPOT fleet.

Q: Your manager asks: should we move from EMR to Databricks?
A: Frame it as a tradeoff analysis:
   Cost: at current scale ($1K/month EMR), Databricks would cost $3-4K/month due to DBU fees.
   Productivity: Databricks notebooks, Unity Catalog, DLT, MLflow reduce engineering time.
   If the team saves 1 engineer-month/quarter ($15K), Databricks pays for itself.
   Technical: Photon engine can be 2-5× faster for SQL workloads.
   Migration risk: Delta Lake format is portable — same Parquet files, same schema.
   Recommend: pilot Databricks for new ML workloads (MLflow + Feature Store).
   Keep EMR for existing high-volume batch (proven, cost-optimized with Spot).
   Reassess in 12 months based on team size and data volume growth.

Q: A business stakeholder reports that yesterday's revenue numbers in QuickSight are wrong.
A: Triage:
   1. Check Airflow: did yesterday's DAG run succeed? Any failed tasks?
   2. Check Gold freshness: SELECT MAX(_updated_at) FROM gold.revenue_by_region
      WHERE date='yesterday'. Was it updated?
   3. Check Silver: SELECT SUM(amount) FROM silver.orders WHERE date='yesterday'.
      Does it match QuickSight's wrong number?
   4. Check Bronze: SELECT COUNT(*) FROM bronze.orders WHERE date='yesterday'.
      Expected volume?
   5. Check DQ metrics: read dq_metrics/silver_2024-01-14.json — any violations?
   If all layers look correct: issue is QuickSight SPICE cache not refreshed.
   Force SPICE refresh. Add alert if QuickSight hasn't refreshed within 3h of Gold update.
""")

---
## Final Review: Key Numbers to Remember

In [ ]:
print("""
Key Numbers Every Data Engineer Should Know
══════════════════════════════════════════════════════════════

SPARK DEFAULTS (know these, explain why you'd change them)
  spark.sql.shuffle.partitions        = 200   (change to ~2-4× executor cores)
  spark.sql.autoBroadcastJoinThreshold = 10MB  (raise to 100MB-1GB for larger dims)
  spark.executor.memory               = 1g    (production: 4-16g)
  spark.driver.memory                 = 1g    (production: 4-8g, more if collect()ing large data)
  target Parquet file size            = 128MB-1GB (OPTIMIZE target in Delta)

DELTA LAKE
  VACUUM default retention  = 7 days
  OPTIMIZE target file size = 1 GB
  Time travel max (default) = 7 days (limited by VACUUM)
  CDF overhead              = ~10-20% write amplification

AWS ROUGH COSTS (know the order of magnitude)
  S3 storage    = $0.023/GB/month → 1 TB = $23/month
  EMR m5.xlarge = $0.096/hr on-demand, ~$0.03/hr Spot (70% savings)
  Athena        = $5/TB scanned → partition pruning is critical
  MSK Kafka     = ~$150/broker/month (3 brokers = $450)
  MWAA Airflow  = ~$500/month (small env)

RULE OF THUMB THRESHOLDS
  GC time > 10%   of task time → executor memory pressure
  Shuffle spill   in Spark UI  → executor memory too small or partitions too large
  Straggler task > 5× median  → data skew
  DQ null rate  > 0.1%        → alert (ShopStream SLA)
  Freshness lag > 2h          → alert (Gold SLA)
  Distribution mean shift > 20% → alert (anomaly)
  Test coverage < 80%         → CI fails

KAFKA
  startingOffsets=latest    → new data only (production restart)
  startingOffsets=earliest  → full replay from topic start
  Watermark for windowed agg = P99 event lateness × 2
  Trigger processingTime    = 5 minutes for 5-min dashboard SLA

AIRFLOW
  mode=reschedule           → always for sensors > 5 min
  trigger_rule=ALL_DONE     → always on terminate cluster task
  max_active_runs=1         → prevent overlapping pipeline runs
  catchup=False             → default for production DAGs
══════════════════════════════════════════════════════════════
""")

---
## Congratulations — Course Complete!

You have completed all 16 weeks of the Apache Spark mastery course:

| Phase | Weeks | Topics |
|-------|-------|--------|
| 1 | 1-2 | Spark fundamentals, RDDs |
| 2 | 3-4 | DataFrames, Spark SQL |
| 3 | 5-7 | Advanced transformations, optimization |
| 4 | 8-9 | PySpark ETL engineering |
| 5 | 10-12 | Structured Streaming, Delta Lake |
| 6 | 13-14 | Cloud platforms, orchestration |
| 7 | 15-16 | Testing, CI/CD, capstone |

### What's Next?

1. **Practice coding questions** — LeetCode, Glassdoor for company-specific patterns
2. **Build something real** — deploy one ShopStream component to AWS
3. **Get Databricks certified** — Databricks Certified Associate Developer for Apache Spark
4. **Interview** — you have everything you need